In [1]:
def clean_data(df):
    '''
    Function cleans the data and prepares it for training.
    Input: dataframe containing product reviews. 
    Output: cleaned dataframe containing relevant columns, 
    with ratings matched to correct labels, duplicates removed, 
    and review text and titles combined into one column. 
    '''
    df_clean = df.copy()

    # Select relevant columns
    df_clean = df_clean[["reviews.text", "reviews.title", "reviews.rating"]]

    # Drop examples where text or title is missing 
    df_clean = df_clean.dropna(subset=["reviews.text"])
    df_clean = df_clean.dropna(subset=["reviews.title"])

    # Map star ratings to categories: 1-2 - negative, 3 - neutral, 4-5 - positive 
    label_map = {
        5: 2, 
        4: 2, 
        3: 1, 
        2: 0, 
        1: 0
    }
    df_clean["label"] = df_clean["reviews.rating"].map(label_map)
    df_clean = df_clean.dropna(subset=["label"])
    df_clean["label"] = df_clean["label"].astype(int)

    # Drop identical reviews 
    df_clean = df_clean.drop_duplicates(subset=["reviews.text"], keep="first")

    # Combine review title and text 
    df_clean["full_text"] = df_clean["reviews.title"] + " " + df_clean["reviews.text"]

    data_clean = df_clean[["full_text", "label"]]
    return data_clean

In [2]:
import pandas as pd 
data = pd.read_csv("../data/1429_1.csv")
data.head()

/var/folders/gn/wg9xjv455tj2c2jns2j4b8yw0000gn/T/ipykernel_53899/3041169091.py:2: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../data/1429_1.csv")


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [3]:
data_clean = clean_data(data)

In [4]:
# Check class balance 
print("-- Positive --")
print(len(data_clean[data_clean["label"] == 2]))
print("-- Neutral --")
print(len(data_clean[data_clean["label"] == 1]))
print("-- Negative --")
print(len(data_clean[data_clean["label"] == 0]))

-- Positive --
32309
-- Neutral --
1499
-- Negative --
812


In [5]:
# Downsampling to handle class imbalance 
df_pos = data_clean[data_clean["label"] == 2]
df_neu = data_clean[data_clean["label"] == 1]
df_neg = data_clean[data_clean["label"] == 0]

df_pos_downsampled = df_pos.sample(n=len(df_neu), random_state=42)
df_balanced = pd.concat([df_pos_downsampled, df_neu, df_neg])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [6]:
df_balanced.head()

,full_text,label
0,Great device for house this device is awesome....,2
1,Disappointed Very disappointed that in order t...,0
2,Worst device Amazon has produced. Over the pas...,0
3,Awesome product worth it would recommend it Aw...,2
4,Not great at all When I first got it was great...,0


In [ ]:
from datasets import Dataset 
from transformers import AutoTokenizer, DataCollatorWithPadding
import evaluate
import numpy as np

raw_dataset = Dataset.from_dict(df_balanced)
split_data = raw_dataset.train_test_split(test_size=0.2, seed=42)


accuracy = evaluate.load("accuracy")

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(batch):
    return tokenizer(batch["full_text"], truncation=True)

tokenized_datasets = split_data.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

Map: 100%|██████████| 3810/3810 [00:00<00:00, 32493.29 examples/s]


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="review_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from transformers import pipeline 

text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."

classifier = pipeline("sentiment-analysis", model="stevhliu/my_awesome_model")
classifier(text)